In [27]:
import numpy as np
from scipy.optimize import curve_fit
import fabio

def gaussian_2d(xy, amplitude, x0, y0, sigma_x, sigma_y, background):
    x, y = xy
    g = amplitude * np.exp(-(((x - x0)**2 / (2 * sigma_x**2)) + ((y - y0)**2 / (2 * sigma_y**2)))) + background
    return g.ravel() 

def removeBeamStopShadow(log_processed):
    #Mask comes from relevant data file
    mask_bool = (fabio.open('/nfs/chess/id4baux/2026-2/wilson-4798-b/calibrations/mask.edf').data == 1)
    
    image = log_processed.copy()
    image[mask_bool] = 0
    image[2425:2550, 0:75] = 0
    
    y_fit, x_fit = np.where(~mask_bool)
    
    z_fit = image[y_fit, x_fit]
    
    #Fit unmasked
    #Initial given based on previous runs
    #Could get x0/y0 from header, but likely won't change efficiency by any notable amount
    popt, pcov = curve_fit(
        gaussian_2d, 
        (x_fit, y_fit),
        z_fit,  
        p0=[-4.61497228e-01, 1224, 1445.003, 3.75145270e+01, 2.83075203e+01, 2.79153325e-01]    , 
        ftol=1e-3,  
        xtol=1e-3,  
        maxfev=2000
    )
    
    amplitude, x0, y0, sigma_x, sigma_y, background = popt

    #To see output, if desired
    #print(f"Fitted Center: ({x0:.2f}, {y0:.2f})")
    #print(f"Fitted Sigmas: x={sigma_x:.2f}, y={sigma_y:.2f}")
    #print(f"Fitted Amplitude: amplitude={amplitude:.2f}")
    #print(f"Fitted Background: background={background:.2f}")

    #Make full grid
    y_size, x_size = mask_bool.shape
    y_full, x_full = np.indices((y_size, x_size))

    # Evaluate the Gaussian model over the full grid 
    gaussian_model = gaussian_2d((x_full, y_full), amplitude, x0, y0, sigma_x, sigma_y, background=0)
    gaussian_model = gaussian_model.reshape(mask_bool.shape)
    
    # Subtract model i.e beamstop shadow out
    image -= gaussian_model

    #Add rectangle mask
    
    return image

In [28]:
import fabio
import os
import numpy as np
import matplotlib.pyplot as plt
import sys
import time
import matplotlib.patches as patches  

start = time.time()

# -------------------------------
# Configuration (Define your folders here)
# -------------------------------
input_folders = ["/nfs/chess/id4b/2026-2/wilson-4798-b/raw6M/CeCa25Cd3P3/CeCa25Cd3P3_b1/300/CeCa25Cd3P3_004"]
output_folder = "/nfs/chess/user/aa2994/backgroundTest"

# Create output folder
try:
    os.makedirs(output_folder, exist_ok=True)
    print(f"Output folder ready: {output_folder}")
except Exception as e:
    print(f"ERROR: Cannot create output folder: {e}")
    sys.exit(1)

processed_log_images = []
folder_metadata = []


# -------------------------------
# Stage 1: Load and Sum CBF Files Per Folder
# -------------------------------
for folder_idx, folder_path in enumerate(input_folders):
    time1 = time.time()
    print(f"\n--- Processing Folder {folder_idx + 1}/{len(input_folders)}: {os.path.basename(folder_path)} ---")
    
    if not os.path.exists(folder_path):
        print(f"ERROR: Input folder does not exist: {folder_path}")
        continue

    try:
        all_files = os.listdir(folder_path)
        cbf_files = sorted([f for f in all_files if f.endswith(".cbf")])[:1]
        print(f"Found {len([f for f in all_files if f.endswith('.cbf')])} CBF files, processing first {len(cbf_files)}")
    except Exception as e:
        print(f"ERROR: Cannot read input folder {folder_path}: {e}")
        continue

    if len(cbf_files) == 0:
        print(f"WARNING: No CBF files found in {folder_path}!")
        continue
    
    sum_image = None
    successful_files = 0
    failed_files = []
    first_three_images = []

    for i, file_name in enumerate(cbf_files):
        try:
            file_path = os.path.join(folder_path, file_name)
            if not os.path.exists(file_path):
                print(f"Warning: File does not exist: {file_name}")
                failed_files.append(file_name)
                continue
            data = fabio.open(file_path).data
            
            img_data = data.astype(np.float32)
                        
            if sum_image is None:
                sum_image = img_data
            else:
                sum_image += img_data
                
            successful_files += 1
            
            if (i + 1) % 5 == 0:
                print(f"  Processed {i + 1}/{len(cbf_files)} files ({successful_files} successful)")
                
        except Exception as e:
            print(f"  Error with {file_name}: {e}")
            failed_files.append(file_name)

    if sum_image is None:
        print(f"WARNING: No valid data loaded for folder {folder_path}!")
        continue

    # Remove negative pixels
    negative_count = np.sum(sum_image < 0)
    if negative_count > 0:
        print(f"  Removing {negative_count:,} negative pixels")
        sum_image = np.maximum(sum_image, 0)

    log_img = np.log1p(sum_image)

    #log_img = removeBeamStopShadow(log_img)    
    
    processed_log_images.append(log_img)
    
    folder_metadata.append({
        'path': folder_path,
        'name': os.path.basename(folder_path),
        'successful_files': successful_files
    })
    
    time2 = time.time()
    print(time2-time1)

intensities = []

# -------------------------------
# Stage 2: Calculate Global Percentile Scaling
# -------------------------------
print("\n--- Calculating Global Contrast Range ---")
global_vmins = []
global_vmaxs = []

for log_img in processed_log_images:
    log_img -= np.mean(log_img)
    vmin_i, vmax_i = np.percentile(log_img, [1, 99.5])
    global_vmins.append(vmin_i)
    global_vmaxs.append(vmax_i)

vmin = min(global_vmins)
vmax = max(global_vmaxs)

print(f"Global vmin: {vmin:.4f}, Global vmax: {vmax:.4f}")

# Clip and normalize all log arrays
normalized_images = [np.clip((log_img - vmin) / (vmax - vmin), vmin, vmax) for log_img in processed_log_images]

# -------------------------------
# Stage 3: Generate Plots
# -------------------------------
print("\n--- Generating Plots ---")

# 1. Save standard log plots for every folder
for idx, norm_img in enumerate(normalized_images):
    meta = folder_metadata[idx]
    
    plt.figure(figsize=(10, 8))
    im = plt.imshow(norm_img, cmap='bwr', origin='lower', vmin=0)
    plt.title(f"{meta['name']},\n{meta['successful_files']} CBF files summed, No Subtraction", fontsize=14)
    plt.colorbar(im, label='Normalized Log Intensity')
    plt.xlabel('Detector X (pixels)')
    plt.ylabel('Detector Y (pixels)')
    
    output_plot = os.path.join(output_folder, f"log_{idx + 1}{time.time()}.png")
    plt.savefig(output_plot, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved standard plot: {output_plot}")
    
end = time.time()

print(f"\nExecution finished in {end - start:.2f} seconds.")


Output folder ready: /nfs/chess/user/aa2994/backgroundTest

--- Processing Folder 1/1: CeCa25Cd3P3_004 ---
Found 3651 CBF files, processing first 1
  Removing 526,973 negative pixels
0.15937113761901855

--- Calculating Global Contrast Range ---
Global vmin: -1.7094, Global vmax: 2.0518

--- Generating Plots ---
Saved standard plot: /nfs/chess/user/aa2994/backgroundTest/log_11783021087.982024.png

Execution finished in 1.88 seconds.


In [29]:
import fabio
import os
import numpy as np
import matplotlib.pyplot as plt
import sys
import time
import matplotlib.patches as patches  

start = time.time()

# -------------------------------
# Configuration (Define your folders here)
# -------------------------------
input_folders = ["/nfs/chess/id4b/2026-2/wilson-4798-b/raw6M/CeCa25Cd3P3/CeCa25Cd3P3_b1/300/CeCa25Cd3P3_004"]
output_folder = "/nfs/chess/user/aa2994/backgroundTest"

# Create output folder
try:
    os.makedirs(output_folder, exist_ok=True)
    print(f"Output folder ready: {output_folder}")
except Exception as e:
    print(f"ERROR: Cannot create output folder: {e}")
    sys.exit(1)

processed_log_images = []
folder_metadata = []


# -------------------------------
# Stage 1: Load and Sum CBF Files Per Folder
# -------------------------------
for folder_idx, folder_path in enumerate(input_folders):
    time1 = time.time()
    print(f"\n--- Processing Folder {folder_idx + 1}/{len(input_folders)}: {os.path.basename(folder_path)} ---")
    
    if not os.path.exists(folder_path):
        print(f"ERROR: Input folder does not exist: {folder_path}")
        continue

    try:
        all_files = os.listdir(folder_path)
        cbf_files = sorted([f for f in all_files if f.endswith(".cbf")])[:1]
        print(f"Found {len([f for f in all_files if f.endswith('.cbf')])} CBF files, processing first {len(cbf_files)}")
    except Exception as e:
        print(f"ERROR: Cannot read input folder {folder_path}: {e}")
        continue

    if len(cbf_files) == 0:
        print(f"WARNING: No CBF files found in {folder_path}!")
        continue
    
    sum_image = None
    successful_files = 0
    failed_files = []
    first_three_images = []

    for i, file_name in enumerate(cbf_files):
        try:
            file_path = os.path.join(folder_path, file_name)
            if not os.path.exists(file_path):
                print(f"Warning: File does not exist: {file_name}")
                failed_files.append(file_name)
                continue
            data = fabio.open(file_path).data
            
            img_data = data.astype(np.float32)
                        
            if sum_image is None:
                sum_image = img_data
            else:
                sum_image += img_data
                
            successful_files += 1
            
            if (i + 1) % 5 == 0:
                print(f"  Processed {i + 1}/{len(cbf_files)} files ({successful_files} successful)")
                
        except Exception as e:
            print(f"  Error with {file_name}: {e}")
            failed_files.append(file_name)

    if sum_image is None:
        print(f"WARNING: No valid data loaded for folder {folder_path}!")
        continue

    # Remove negative pixels
    negative_count = np.sum(sum_image < 0)
    if negative_count > 0:
        print(f"  Removing {negative_count:,} negative pixels")
        sum_image = np.maximum(sum_image, 0)

    log_img = np.log1p(sum_image)

    log_img = removeBeamStopShadow(log_img)    
    
    processed_log_images.append(log_img)
    
    folder_metadata.append({
        'path': folder_path,
        'name': os.path.basename(folder_path),
        'successful_files': successful_files
    })
    
    time2 = time.time()
    print(time2-time1)

intensities = []

# -------------------------------
# Stage 2: Calculate Global Percentile Scaling
# -------------------------------
print("\n--- Calculating Global Contrast Range ---")
global_vmins = []
global_vmaxs = []

for log_img in processed_log_images:
    log_img -= np.mean(log_img)
    vmin_i, vmax_i = np.percentile(log_img, [1, 99.5])
    global_vmins.append(vmin_i)
    global_vmaxs.append(vmax_i)

vmin2 = min(global_vmins)
vmax2 = max(global_vmaxs)

print(f"Global vmin: {vmin2:.4f}, Global vmax: {vmax2:.4f}")

#Using old one to have easy image-by-image comparison
print("NOTE: OLD VMIN, VMAX USED")

# Clip and normalize all log arrays
normalized_images2 = [np.clip((log_img - vmin) / (vmax - vmin), vmin, vmax) for log_img in processed_log_images]

# -------------------------------
# Stage 3: Generate Plots
# -------------------------------
print("\n--- Generating Plots ---")

# 1. Save standard log plots for every folder
for idx, norm_img in enumerate(normalized_images2):
    meta = folder_metadata[idx]
    
    plt.figure(figsize=(10, 8))
    im = plt.imshow(norm_img, cmap='bwr', origin='lower', vmin=0)
    plt.title(f"{meta['name']},\n{meta['successful_files']} CBF files summed, Subtracted Image", fontsize=14)
    plt.colorbar(im, label='Normalized Log Intensity')
    plt.xlabel('Detector X (pixels)')
    plt.ylabel('Detector Y (pixels)')
    
    output_plot = os.path.join(output_folder, f"log_{idx + 1}{time.time()}.png")
    plt.savefig(output_plot, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved standard plot: {output_plot}")
end = time.time()

print(f"\nExecution finished in {end - start:.2f} seconds.")

Output folder ready: /nfs/chess/user/aa2994/backgroundTest

--- Processing Folder 1/1: CeCa25Cd3P3_004 ---
Found 3651 CBF files, processing first 1
  Removing 526,973 negative pixels
16.00192880630493

--- Calculating Global Contrast Range ---
Global vmin: -2.8999, Global vmax: 1.0737
NOTE: OLD VMIN, VMAX USED

--- Generating Plots ---
Saved standard plot: /nfs/chess/user/aa2994/backgroundTest/log_11783021105.759443.png

Execution finished in 17.78 seconds.


In [30]:
idx= 0
#Normalized images2 is the subtracted image
norm_img1 = normalized_images[0]
norm_img2 = normalized_images2[0]

log_processed_diff = norm_img1 - norm_img2

plt.figure(figsize=(10, 8))
im = plt.imshow(log_processed_diff, cmap='bwr', origin='lower')
plt.title("Difference", fontsize=14)
plt.colorbar(im, label='Normalized Log Intensity')
plt.xlabel('Detector X (pixels)')
plt.ylabel('Detector Y (pixels)')

output_diff_plot = os.path.join(output_folder, f"Comparison{time.time()}.png")
plt.savefig(output_diff_plot, dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print(f"Saved difference plot: {output_diff_plot}")

Saved difference plot: /nfs/chess/user/aa2994/backgroundTest/Comparison1783021107.4232872.png
